# Did the molecule really split in two? Count the peaks in a carbon spectrum

You'll check the notebook 01 peptides again, this time through their carbon atoms, and count the
large carbonyl peaks before and after. This notebook goes with the NMR project Extra of the same name.

Hydrogen NMR (1H) shows the hydrogens; carbon NMR (13C) shows the carbon skeleton. In these carbon
spectra, read which peaks are there, not their size. They were recorded fast, with about one second
between scans. A carbon with no hydrogens attached, such as a carbonyl (C=O), recovers slowly
between scans and comes out weak.

Time: about 30 minutes. Same samples as notebook 01, in 98% acid. Boxes headed **RECORD YOUR
ANSWER** are where you write something; you compare with the paper at the end.

Data: Petkowski, M. D. Seager, Bains and S. Seager, Zenodo [10.5281/zenodo.11223995](https://doi.org/10.5281/zenodo.11223995), CC BY 4.0. Paper: *Scientific Reports* 14, 17083 (2024).

In [ ]:
# Setup. In Colab this first installs the training package and the Rakiura helper (about 30 s).
import importlib.util

if importlib.util.find_spec("nmrtrain") is None:
    %pip install -q "nmrtrain @ git+https://github.com/Rakiura-Inc/morning-star-nmr@main"
if importlib.util.find_spec("rakiura") is None:
    %pip install -q "rakiura @ git+https://github.com/Rakiura-Inc/rakiura-py@main"
import matplotlib.pyplot as plt
import nmrglue as ng  # the standard Python library for reading and processing NMR data
import numpy as np
from scipy.signal import find_peaks

import nmrtrain as nt  # this course's plumbing: downloads, checks, your answers, the hand-in

NOTEBOOK = "notebooks/05_read_a_13c_series.ipynb"
MODE = nt.portal.connect(NOTEBOOK)

> **CONNECT TO RAKIURA (optional, recommended)**
>
> **What it does:** ticks your checklist on the step as you go and attaches your figures to it. Nothing is sent until you connect.
>
> **How:** run the cell below, open the link it prints, check the code matches the one printed here, and click Allow. In Colab you do this once per session. To skip the click, add your key as a Colab secret named `RAKIURA_KEY` (the key icon on the left; get the key in Rakiura under Settings > Notebook key).
>
> **Skipping it is fine:** your work is kept on this computer, and you can attach your files on the step page instead.

In [ ]:
# Connect to Rakiura (optional). Open the link it prints, check the code matches, click Allow.
MODE = nt.portal.connect(NOTEBOOK, sign_in=True)

### What you need

Ala-Ala has three kinds of carbon, two of each (one per alanine), coloured in the drawing below:

- the carbonyls (C=O): one in the peptide bond, one in the carboxylic acid group (COOH) at the end
  of the chain
- the alpha carbons: the backbone carbons next to the nitrogen
- the methyl carbons (CH3)

Typical positions in carbon NMR:

| Carbon | Typical position |
|---|---|
| carbonyl (C=O) | near 170 to 180 ppm |
| carbon bonded to nitrogen | about 30 to 65 ppm |
| methyl carbon | about 10 to 30 ppm |

A carbon bonded to oxygen or nitrogen sits at higher ppm than one bonded only to carbon and
hydrogen.

In [ ]:
# Ala-Ala's carbons by kind: orange carbonyls (C=O), blue alpha carbons, green methyl carbons.
carbonyl, alpha, methyl = nt.plot.PALE_ORANGE, nt.plot.PALE_BLUE, nt.plot.PALE_GREEN
carbon_kinds = {"C1": ("", carbonyl), "C2": ("", carbonyl)}  # C: the carbonyl carbons
carbon_kinds.update({"A1": ("", alpha), "A2": ("", alpha)})  # A: the alpha carbons
carbon_kinds.update({"S1": ("", methyl), "S2": ("", methyl)})  # S: the side-chain methyls
fig, ax = plt.subplots(figsize=(7, 3.2))
nt.plot.draw_dipeptide(ax, "Ala", "Ala-Ala: three kinds of carbon", mark=carbon_kinds)
plt.show()

> **RECORD YOUR ANSWER: predict where each kind of carbon appears**
>
> **What gets saved:** `MY_PREDICTION`, three positions in ppm and two counts, saved with the time.
>
> **Where to look:** the drawing and the table of typical positions above.
>
> **How to fill this in:**
> - `carbonyl_ppm`: the carbonyls (C=O), a number in ppm like `100`
> - `alpha_ppm`: the alpha carbons
> - `methyl_ppm`: Ala-Ala's methyl carbons
> - `carbonyls_1day` and `carbonyls_4months`: how many large carbonyl peaks you expect at each time, whole numbers
> - The cell after it says where these kinds of carbon typically sit; you then find the real positions in the stack.

<details><summary>Stuck? Open hint 1</summary>

Start with the carbonyl: the table gives its region.

</details>

<details><summary>Stuck? Open hint 2</summary>

The more oxygen or nitrogen on or next to a carbon, the higher its ppm: the carbonyl highest,
the alpha carbon (bonded to nitrogen) next, the methyl lowest.

</details>

In [ ]:
MY_PREDICTION = {
    "carbonyl_ppm": ...,  # a number in ppm: the C=O carbons
    "alpha_ppm": ...,  # a number in ppm: the backbone carbons next to the nitrogen
    "methyl_ppm": ...,  # a number in ppm: Ala-Ala's CH3 carbons
    "carbonyls_1day": ...,  # a whole number: the large carbonyl peaks you expect at 1 day
    "carbonyls_4months": ...,  # a whole number: the large carbonyl peaks you expect at 4 months
}
nt.record.predict("c13", MY_PREDICTION)

In [ ]:
# Reveal: runs only after you predicted. It checks your reasoning, not this dataset.
if nt.record.revealed("c13"):
    c13_key = nt.answers.load("c13_series", quiet=True)
    prediction_feedback = nt.checks.check_prediction(MY_PREDICTION, c13_key["prediction"])
    nt.checks.show(prediction_feedback, "Your predictions")

## Load and stack

In [ ]:
# How every spectrum in this notebook is read, with nmrglue.
def read_spectrum(zip_path):
    """The spectrum the instrument processed, with its ppm axis and labels (sample, acid, time)."""
    folder = nt.io.experiment_folder(zip_path)  # unzipped once, into nmr_data/
    # Bruker keeps the spectrum it processed in pdata/1; dic holds the instrument's settings.
    dic, data = ng.bruker.read_pdata(f"{folder}/pdata/1")
    # udic: the axis settings (number of points, spectral width, spectrometer frequency) from dic.
    udic = ng.bruker.guess_udic(dic, data)
    ppm = ng.fileiobase.uc_from_udic(udic).ppm_scale()  # the instrument's ppm scale, high to low
    return nt.io.labelled(zip_path, dic, ppm, data)

In [ ]:
DIPEPTIDE_SPECTRA = "11223995"  # Zenodo record: Petkowski et al., CC BY 4.0
ala_ala_files = nt.data.fetch(DIPEPTIDE_SPECTRA, "Ala-Ala", acid="98%", nucleus="13C")
gly_gly_files = nt.data.fetch(DIPEPTIDE_SPECTRA, "Gly-Gly", acid="98%", nucleus="13C")
ala_ala_13c = [read_spectrum(path) for path in ala_ala_files]  # first time point first
gly_gly_13c = [read_spectrum(path) for path in gly_gly_files]

In [ ]:
def scaled(spectrum, scale_region):
    """The intensity divided by its tallest point inside scale_region (low, high ppm), so that point
    is 1. With scale_region None, the tallest point anywhere (in a hydrogen spectrum in this acid,
    the acid's own peak)."""
    if scale_region is None:
        return spectrum.intensity / spectrum.intensity.max()
    low_ppm, high_ppm = scale_region
    _, intensity_in_region = spectrum.region(low_ppm, high_ppm)  # region gives (ppm, intensity)
    return spectrum.intensity / intensity_in_region.max()

In [ ]:
def plot_stack(spectra, ppm_range, *, scale_region, title, labels=None, ax=None, offset=1.2):
    """Traces stacked bottom to top in the order given, each scaled by scaled(), then raised by
    offset. Everything after the * is given by name: plot_stack(spectra, (0, 13), title=...)."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 5))
    colours = plt.cm.viridis(np.linspace(0.0, 0.85, len(spectra)))  # dark to light, short of yellow
    for number, spectrum in enumerate(spectra):
        label = labels[number] if labels else spectrum.time_label
        trace = scaled(spectrum, scale_region) + offset * number
        ax.plot(spectrum.ppm, trace, lw=0.8, color=colours[number], label=label)
    ax.set_xlim(ppm_range[1], ppm_range[0])  # NMR convention: high ppm on the left
    ax.set_ylim(-0.2, offset * (len(spectra) - 1) + 1.4)
    ax.set_yticks([])  # the heights are scaled, so their numbers mean nothing
    ax.set_xlabel("chemical shift (ppm)")
    ax.set_title(title)
    handles, names = ax.get_legend_handles_labels()  # listed top trace first, like the stack
    ax.legend(handles[::-1], names[::-1], frameon=False, fontsize=8, loc="upper left")
    return ax.figure

In [ ]:
# From 0 to 200 ppm, each trace scaled to its tallest point from 5 ppm up.
fig_13c, (ax_ala_ala, ax_gly_gly) = plt.subplots(1, 2, figsize=(14, 5.5))
title = "Ala-Ala in 98% acid, 13C"
plot_stack(ala_ala_13c, ppm_range=(0, 200), scale_region=(5, 200), title=title, ax=ax_ala_ala)
traps_ala_ala = nt.checks.run_traps(ala_ala_13c)
title = "Gly-Gly in 98% acid, 13C"
plot_stack(gly_gly_13c, ppm_range=(0, 200), scale_region=(5, 200), title=title, ax=ax_gly_gly)
traps_gly_gly = nt.checks.run_traps(gly_gly_13c)
plt.show()

### Trap: missing carbonyls

Look at Gly-Gly near 170 ppm: its carbonyls show in only one spectrum. In these fast carbon runs a
carbonyl peak can be missing from a whole spectrum, so do not count a carbonyl as gone just because
it is missing. Use the spectra where the carbonyls do show: count the large ones at each time point,
and check whether each sits where it did at 1 day.

In [ ]:
def group_lines(lines, group_ppm):
    """Lines (ppm, height), low ppm first, closer than group_ppm to the line before belong to the
    same signal: a methyl's two lines are 0.016 ppm apart."""
    groups = []
    previous_ppm = None
    for ppm, height in lines:
        if previous_ppm is not None and ppm - previous_ppm < group_ppm:
            groups[-1].append((ppm, height))
        else:
            groups.append([(ppm, height)])
        previous_ppm = ppm
    return groups

In [ ]:
def signal_from(lines):
    """One signal from its lines: their height-weighted centre, how many, and the tallest."""
    total_height = sum(height for _, height in lines)
    centre_ppm = sum(ppm * height for ppm, height in lines) / total_height
    tallest = max(height for _, height in lines)
    return {
        "ppm": round(float(centre_ppm), 3),
        "lines": len(lines),
        "height": round(float(tallest), 3),
    }

In [ ]:
def pick_peaks(spectrum, ppm_range, *, min_height, group_ppm=0.03, scale_region=None):
    """The signals between two ppm values, high ppm first, each {"ppm", "lines", "height"}.
    min_height and the heights are shares of the tallest point in ppm_range, or in scale_region
    when you give one (to look for small peaks in a quiet window)."""
    ppm, intensity = spectrum.region(*ppm_range)  # the (low, high) pair, unpacked
    if scale_region is None:
        heights = intensity / intensity.max()
    else:
        _, intensity_in_region = spectrum.region(*scale_region)
        heights = intensity / intensity_in_region.max()
    # A line must reach min_height and stand 0.02 above the dips beside it (ripples do not).
    found, _ = find_peaks(heights, height=min_height, prominence=0.02)
    lines = sorted(zip(ppm[found], heights[found]))  # (ppm, height) pairs, low ppm first
    signals = [signal_from(group) for group in group_lines(lines, group_ppm)]
    return signals[::-1]  # high ppm first, the way a spectrum is read

In [ ]:
MIN_HEIGHT = 0.05  # the smallest peak listed, as a share of the tallest
GROUP_PPM = 0.5  # carbon lines closer than half a ppm count as one peak

# Carbonyls between 160 and 185 ppm, as a share of the tallest peak from 5 ppm up, so a quiet
# window does not turn its noise into peaks. Lines within half a ppm count as one peak.
for spectrum in gly_gly_13c:
    peaks = pick_peaks(
        spectrum, (160, 185), min_height=MIN_HEIGHT, group_ppm=GROUP_PPM, scale_region=(5, 200)
    )
    seen = [peak["ppm"] for peak in peaks]
    if seen:
        print(f"Gly-Gly {spectrum.time_label:9s} carbonyl peaks: {seen}")
    else:
        print(f"Gly-Gly {spectrum.time_label:9s} carbonyl peaks: none seen")

## Compare first and last

The next cell lists the peaks in the first and last spectrum of each peptide, and which are at
both times, at the first only, and at the last only. `MIN_HEIGHT` sets the smallest peak it lists.

In [ ]:
def carbon_peaks_ppm(spectrum):
    """Where the carbon peaks between 5 and 200 ppm sit."""
    peaks = pick_peaks(spectrum, (5, 200), min_height=MIN_HEIGHT, group_ppm=GROUP_PPM)
    return [peak["ppm"] for peak in peaks]

In [ ]:
SAME_PEAK_PPM = 0.05  # two peaks this close, at two times, are the same peak


def compare_peaks(first_ppm, last_ppm, tolerance_ppm=SAME_PEAK_PPM):
    """Which peaks are at both times (kept), at the first only (lost) and at the last only
    (new). Two peaks within tolerance_ppm of each other count as the same peak."""
    changes = {"kept": [], "lost": [], "new": []}
    for ppm in first_ppm:
        if any(abs(ppm - later) <= tolerance_ppm for later in last_ppm):
            changes["kept"].append(ppm)
        else:
            changes["lost"].append(ppm)
    for ppm in last_ppm:
        if not any(abs(ppm - earlier) <= tolerance_ppm for earlier in first_ppm):
            changes["new"].append(ppm)
    return changes

In [ ]:
def compare_carbon(first_ppm, last_ppm):
    """Kept, lost and new carbon peaks, to one decimal. The same carbon peak at two times sits
    within half a ppm in these spectra."""
    changes = compare_peaks(first_ppm, last_ppm, tolerance_ppm=0.5)
    rounded = {}
    for kind, positions in changes.items():
        rounded[kind] = [round(ppm, 1) for ppm in positions]
    return rounded


def listed(positions_ppm):
    """Positions to one decimal, as '176.5, 50.1', or 'none'."""
    if not positions_ppm:
        return "none"
    return ", ".join(f"{ppm:.1f}" for ppm in positions_ppm)

In [ ]:
ala_ala_first, ala_ala_last = carbon_peaks_ppm(ala_ala_13c[0]), carbon_peaks_ppm(ala_ala_13c[-1])
gly_gly_first, gly_gly_last = carbon_peaks_ppm(gly_gly_13c[0]), carbon_peaks_ppm(gly_gly_13c[-1])
ala_ala_changes = compare_carbon(ala_ala_first, ala_ala_last)
gly_gly_changes = compare_carbon(gly_gly_first, gly_gly_last)

In [ ]:
for name, spectra, first_ppm, last_ppm, changes in (
    ("Ala-Ala", ala_ala_13c, ala_ala_first, ala_ala_last, ala_ala_changes),
    ("Gly-Gly", gly_gly_13c, gly_gly_first, gly_gly_last, gly_gly_changes),
):
    print(name, f"{spectra[0].time:7s}", [round(ppm, 1) for ppm in first_ppm])
    print(name, f"{spectra[-1].time:7s}", [round(ppm, 1) for ppm in last_ppm])
    kept, lost, new = listed(changes["kept"]), listed(changes["lost"]), listed(changes["new"])
    print(f"   at both times {kept} | first only {lost} | last only {new}")

### What you need

Read Ala-Ala's first (1 day) list. Each number is one carbon peak's position in ppm. Use the
table in the first box to say which peak is which kind of carbon.

> **RECORD YOUR ANSWER: where do Ala-Ala's carbons really sit?**
>
> **What gets saved:** `MY_READING`, three positions in ppm, read off Ala-Ala's 1 day list.
>
> **Where to look:** the Ala-Ala 1 day list printed above, and the left stack.
>
> **How to fill this in:**
> - `carbonyl_ppm`: one of the carbonyl peaks, a number like `100.0`
> - `alpha_ppm`: one of the alpha carbon peaks
> - `methyl_ppm`: the methyl carbon peak
> - The check after it compares your reading with the ranges in this data and with your prediction.

<details><summary>Stuck? Open hint 1</summary>

Sort the 1 day list into three groups using the table of typical positions.

</details>

<details><summary>Stuck? Open hint 2</summary>

The highest values are carbonyls. The values between about 40 and 55 are alpha carbons. The
lowest value is the methyl. Leave out anything that fits none of the three.

</details>

In [ ]:
MY_READING = {
    "carbonyl_ppm": ...,  # a number in ppm, from Ala-Ala's 1 day list
    "alpha_ppm": ...,  # a number in ppm
    "methyl_ppm": ...,  # a number in ppm
}
reading_key = nt.answers.load("c13_series", quiet=True)["reading"]
reading_feedback = nt.checks.check_reading(MY_READING, reading_key, MY_PREDICTION)
nt.checks.show(reading_feedback, "Where Ala-Ala's carbons sit at 1 day")

### A filled example: Gly-Gly

Gly-Gly, filled in with the reasons. It shows the format; you do Ala-Ala below.

In [ ]:
GLY_GLY_EXAMPLE = {
    # its carbon peaks stay in place, as its hydrogen signals did in notebook 01
    "call": "survives",
    # nothing gone (its carbonyls show in only one spectrum: not evidence they are gone)
    "lost_ppm": [],
    "new_ppm": [],  # nothing new
    "kept_ppm": [44.2, 41.3],  # the two CH2 carbons, one per glycine
    # carbon supports the hydrogen decision; these fast 13C runs miss weak peaks
    "confidence": "medium",
}

### What you need

- **Lost:** a peak at 1 day that is gone at the last time point.
- **New:** a peak at the last time point that was not there at 1 day. If Ala-Ala breaks down, expect
  many small new peaks: in 98% acid it releases acrylic acid, which keeps reacting into many products.
- **Kept:** a peak at both times.
- Peak heights in these carbon spectra do not measure amounts, so read which peaks are there, not
  their size.

> **RECORD YOUR ANSWER: what changed in Ala-Ala's carbon spectrum?**
>
> **What gets saved:** `AA_LOST_PPM`, `AA_NEW_PPM` and `AA_KEPT_PPM`, three lists of 13C positions in ppm.
>
> **Where to look:** the Ala-Ala lists printed under Compare first and last, and the left stack.
>
> **How to fill this in:**
> - `AA_LOST_PPM`: peaks there at 1 day and gone at the last time point, as numbers like `[12.3, 45.6]`
> - `AA_NEW_PPM`: peaks at the last time point that were not there at 1 day
> - `AA_KEPT_PPM`: peaks there at both times
> - Leave a list as `[]` if there were none. Gly-Gly's record above shows the format.

<details><summary>Stuck? Open hint 1</summary>

Go peak by peak through the 1 day list: is there still a peak within about half a ppm at the last time point?

</details>

<details><summary>Stuck? Open hint 2</summary>

Count the carbonyls first: at 1 day there are two large ones. How many large ones are there at the
last time point, and does each sit where a 1 day one did?

</details>

In [ ]:
AA_LOST_PPM = []  # peaks there at 1 day and gone at the last time point, like [12.3, 45.6]
AA_NEW_PPM = []  # peaks at the last time point that were not there at 1 day
AA_KEPT_PPM = []  # peaks there at both times

who = "Ala-Ala (13C)"
nt.record.write_down(
    "AA_LOST_PPM", AA_LOST_PPM, who, "lost_ppm", suggestion=ala_ala_changes["lost"]
)
nt.record.write_down("AA_NEW_PPM", AA_NEW_PPM, who, "new_ppm", suggestion=ala_ala_changes["new"])
nt.record.write_down(
    "AA_KEPT_PPM", AA_KEPT_PPM, who, "kept_ppm", suggestion=ala_ala_changes["kept"]
)

> **RECORD YOUR ANSWER: your decision on Ala-Ala from its carbons**
>
> **What gets saved:** `AA_CALL`, `AA_CONFIDENCE` and `AA_NOTE`, three short answers in quotes.
>
> **Where to look:** the lost, new and kept peaks you saved above.
>
> **How to fill this in:**
> - `AA_CALL`: your decision, one of these:
>   - `"survives"`: every large peak stays where it was at 1 day
>   - `"breaks down"`: large peaks vanish or a new large one appears at another position, with many small new peaks
>   - `"unsure"`: the data do not settle it; say why in `AA_NOTE`
> - `AA_CONFIDENCE`: how sure you are: `"low"`, `"medium"` or `"high"`
> - `AA_NOTE`: how many large carbonyl peaks you count at 1 day and at 4 months, what that says about how Ala-Ala broke apart, and, once you have read the paper below, which bond it says breaks

<details><summary>Stuck? Open hint 1</summary>

Count the large carbonyl peaks at 1 day and at 4 months, and compare their positions.

</details>

<details><summary>Stuck? Open hint 2</summary>

Ala-Ala's two carbonyls sit in the peptide bond and in the COOH at the end. In 98% acid the end
alanine can be cut off at its nitrogen: the first alanine then leaves as alaninamide, whose one
carbonyl sits in its new amide group (CONH2), and the end alanine's carbons spread over many small
peaks. So two large carbonyls at 1 day and one large one at a new position later point to breakdown.

</details>

In [ ]:
AA_CALL = ""  # your decision: "survives", "breaks down" or "unsure"
AA_CONFIDENCE = ""  # "low", "medium" or "high"
AA_NOTE = ""  # the large carbonyl peaks you count at 1 day and at 4 months, and what that says

nt.record.write_down("AA_CALL", AA_CALL, who, "call")
nt.record.write_down("AA_CONFIDENCE", AA_CONFIDENCE, who, "confidence")
nt.record.write_down("AA_NOTE", AA_NOTE, who, "note")

### Your verdict

Your verdict is everything you recorded about Ala-Ala's carbons, in one table: your decision, the
peaks behind it, how sure you are and your note. To change it, edit a line above and run this cell
again. The Gly-Gly line is the filled example.

In [ ]:
verdicts = nt.record.build_verdict(
    {
        "AA": {
            "call": AA_CALL,
            "lost_ppm": AA_LOST_PPM,
            "new_ppm": AA_NEW_PPM,
            "kept_ppm": AA_KEPT_PPM,
            "confidence": AA_CONFIDENCE,
            "note": AA_NOTE,
        }
    },
    names={"AA": "Ala-Ala (13C)"},
    examples={"Gly-Gly (13C, example)": GLY_GLY_EXAMPLE},
)
MY_VERDICT = verdicts["AA"]  # this notebook's one case

The next cell compares your decision with the published finding (Petkowski et al. 2024, *Scientific
Reports* 14, 17083). If it differs, look again at the carbonyls; your hand-in goes to review as you
wrote it.

In [ ]:
feedback = []
if nt.record.revealed("c13"):
    key = nt.answers.load("c13_series")
    published = nt.answers.case(key, "AA")
    feedback = nt.checks.check_verdict(MY_VERDICT, published)
    nt.checks.show(feedback, "Your Ala-Ala verdict")

### Read the mechanism paper

The lab tested Ala-Ala, Gly-Gly, Gly-Ala and Ala-Gly in 98% acid. Gly-Ala has glycine first and
alanine at the end of the chain (the end with the free acid group); Ala-Gly is the reverse. Before
you read, predict which two last for months.

Then read [Mechanism for peptide bond solvolysis in 98% w/w concentrated sulfuric acid](https://pmc.ncbi.nlm.nih.gov/articles/PMC11904706/)
(open full text; w/w means by weight). Solvolysis means being broken apart by the solvent, here the
acid itself, rather than by water. Data: [Zenodo 11122291](https://zenodo.org/records/11122291).
Look for which bond breaks, what happens to the alanine at the end of the chain, and what the
products tell you. Add the bond to `AA_NOTE` above and run the cells from there down again.

<details><summary>Check your reading: open it after you have added the bond to AA_NOTE</summary>

**How Ala-Ala breaks in 98% acid**

The lab's mechanism paper (Petkowski et al. 2025, *ACS Omega* 10, 9623) reports how Ala-Ala breaks
in 98% acid. The acid attacks the second alanine, the one at the end of the chain: the acid removes
hydrogen from its methyl group (dehydrogenation), and then the bond between its alpha carbon and its
nitrogen breaks. This is not simple splitting of the peptide bond by water. The first alanine leaves
as alaninamide: alanine with an amide group, C(=O)NH2, where its carboxylic acid group would be. The
second alanine leaves as acrylic acid, which keeps reacting into a mixture of many products.

</details>

## Hand-in

> **SAVE YOUR HAND-IN: your carbon hand-in**
>
> **What gets saved:** in `handin/`: your 13C stacks and `verdict.json` (your verdict table and the feedback).
>
> **Where to look:** nothing new to fill in: the cell uses what you recorded above.
>
> **How to fill this in:**
> - Run the cell. These files are your hand-in for the NMR project Extra, Did the molecule really split in two? The stacks are your image of the carbon spectra, and `verdict.json` holds your note.
> - Nothing is sent for review until you run `nt.portal.submit()` in the last line of the cell.
> - In Colab: open the folder icon on the left, then `handin/c13-verdict`, and download each file before you close the tab.

<details><summary>Stuck? Open hint 1</summary>

If the figure is missing from the list it prints, run the stack cell, then this cell again.

</details>

<details><summary>Stuck? Open hint 2</summary>

If it says a name is not defined, run the notebook from the top (Runtime > Run all in Colab).

</details>

In [ ]:
record = nt.record.handin(
    "c13-verdict",
    figures={"stack_13C": fig_13c},
    verdict={"verdict": MY_VERDICT, "reading": MY_READING, "feedback": feedback},
    numbers={"AA_call_13C": MY_VERDICT["call"]},
    question="c13",
    traps=nt.checks.traps_flagged(traps_ala_ala, traps_gly_gly),
)
# nt.portal.submit()